# Analitica de Datos para Sitio Web E-Commerce Cafetería Cafe-Cereza #

### Etapa 3: Limpieza del Dataset ###

Elaborado por: <br>
1. José de Jesús Hernández Casiano
2. Michelle de la Cruz Rosalino
3. Yuleni Gayosso Martinez

Grado/Grupo: 9A IEVND <br>
<br>
**1. Importamos las librerias de Python para la manipulación y análisis de datos**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as mat
from datetime import datetime
import seaborn as sea
import hashlib
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Extracción del Dataset**

Se manda a traer el dataset directo desde Google Drive, para posteriormente crear el *DataFrame*

In [ ]:
df_csv = pd.read_csv('/content/drive/MyDrive/BD_Cafeteria/dataset/etapa_3/dataset_ventas_sucio.csv')
df_csv

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
0,1018,1415,85,Renata Díaz,NaN,06-Jun-26,Página web,Café frío,Café helado,1,...,entregao,pago en línea,pagado,126.84,20.29,147.13,31.0,5.0,VIP,True
1,991,1402,198,Adrián Reyes,adrián.reyes198@correo.com,2025-06-05,Presencial (mostrador),Café caliente,Latte,1,...,entregado,tarjeta,pagado,288.31,46.13,334.44,27.0,4.0,Frecuente,False
2,765,1314,178,María Pérez,maría.pérez178@correo.com,2025-12-04,Página web,Bebidas Caliente,Infusión de manzanilla,1,...,entregado,pago en línea,pagado,148.34,23.73,172.07,9.0,5.0,Frecuente,True
3,1986,1801,54,Iván S,iván.sánchez54@correo.com,01-Mar-25,presencial mostrador,Café frío,Cold brew,3,...,entregado,tarjeta,pagado,282.21,45.15,327.36,9.0,4.0,VIP,False
4,1273,1515,32,Camila Ramírez,camila.ramírez32@correo.com,2025-03-01,Presencial (mesero),Bebidas frías,Té helado,1,...,entregado,tarjeta,pagado,225.38,36.06,261.44,22.0,4.0,VIP,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3936,1131,1460,20,Lucía López,NaN,2025-09-29,Página web,Alimentos,Bagel con queso crema,3,...,entregado,pago en línea,pagado,671.39,107.42,778.81,9.0,5.0,Frecuente,True
3937,1295,1522,155,Jorge Ortiz,jorge.ortiz155@correo.com,2025-04-16,Página web,Bebidas frías,Jugo de naranja,3,...,entregado,transferencia,pagado,398.35,63.74,462.09,22.0,NaN,Frecuente,True
3938,861,1350,183,Regina Cruz,regina.cruz183@correo.com,2025-02-22,Página web,Café caliente,Capuchino,3,...,listo,pago en línea,pagado,343.43,54.95,398.38,9.0,NaN,Frecuente,True
3939,3508,2414,248,Estefanía García,estefanía.garcía248@correo.com,2025-03-07,Presencial (mesero),Café Caliente,Americano,2,...,entregado,efectivo,pagado,290.79,46.53,337.32,13.0,3.0,Frecuente,False


**3. Registro de Metadatos**

Se lleva a cabo un registro de los siguientes datos:
1. Nombre de la fuente.
2. Formato.
3. Fecha de extracción.
4. Cantidad de registros.
5. Campos disponibles.
6. Problemas encontrados.

In [ ]:
log_extraccion = []

def registrar_fuente(nombre, formato, df, problemas="Sin problemas"):
    log_extraccion.append({
        "Nombre de la fuente": nombre,
        "Formato": formato,
        "Fecha de extracción": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Cantidad de registros": len(df),
        "Campos disponibles": list(df.columns),
        "Problemas encontrados": problemas
    })

registrar_fuente("Ventas CSV", "CSV", df_csv)

**4. Crear un Dataframe con el log**

Este comando nos permite cargar en memoria cahce, los datos de la cafetería que se van a analizar utilizando la libreria **PANDAS** permitiendo la manipulación de datos. <br>

In [ ]:
df_registro_metadatos = pd.DataFrame(log_extraccion)
df_registro_metadatos

,Nombre de la fuente,Formato,Fecha de extracción,Cantidad de registros,Campos disponibles,Problemas encontrados
0,Ventas CSV,CSV,2026-08-20 04:53:59,3941,"[id_venta, id_pedido, id_cliente, nombre_clien...",Sin problemas


**5. Perfilado Inicial de Datos**

Se perfilan los datos por columna, llevando el conteo de valores existentes en cada una de ellas. Donde se revisan los siguientes parametros:
1. Número de filas y columnas.
2. Tipos de datos.
3. Valores nulos.
4. Duplicados.
5. Valores únicos.
6. Rangos.
7. Fechas mínimas y máximas.
8. Categorías existentes.

In [ ]:
def perfilado_inicial(df):
    print("=== 1. FILAS Y COLUMNAS ===")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}\n")

    print("=== 2. TIPOS DE DATOS Y NULOS ===")
    resumen = pd.DataFrame({
        'Tipo': df.dtypes,
        'Nulos': df.isnull().sum(),
        '% Nulos': (df.isnull().sum() / len(df)) * 100,
        'Valores Únicos': df.nunique()
    })
    print(resumen, "\n")

    print("=== 3. DUPLICADOS ===")
    print(f"Total de filas duplicadas: {df.duplicated().sum()}\n")

    print("=== 4. RANGOS Y ESTADÍSTICA ===")
    print(df.describe(include='all'), "\n")

    print("=== 5. FECHAS MÍNIMAS Y MÁXIMAS ===")
    cols_fecha = df.select_dtypes(include=['datetime', 'datetime64']).columns
    for col in cols_fecha:
        print(f"{col}: Mín = {df[col].min()} | Máx = {df[col].max()}")

    print("\n=== 6. CATEGORÍAS EN COLUMNAS TEXTO ===")
    cols_cat = df.select_dtypes(include=['object', 'category']).columns
    for col in cols_cat:
        print(f"\nColumna '{col}':\n{df[col].value_counts().head(5)}")

perfilado_inicial(df_csv)

=== 1. FILAS Y COLUMNAS ===
Filas: 3941 | Columnas: 31

=== 2. TIPOS DE DATOS Y NULOS ===
                           Tipo  Nulos    % Nulos  Valores Únicos
id_venta                  int64      0   0.000000            3754
id_pedido                 int64      0   0.000000            1500
id_cliente                int64      0   0.000000             250
nombre_cliente           object     85   2.156813             272
correo_cliente           object    353   8.957117             254
fecha_registro_cliente   object      0   0.000000             531
canal_venta              object      0   0.000000              14
categoria_producto       object      0   0.000000              29
producto                 object     82   2.080690              33
cantidad                  int64      0   0.000000               7
precio_unitario         float64    280   7.104796            2609
subtotal_linea          float64      0   0.000000            3171
tipo_pedido              object      0   0.000000   

**6. Leer los datos de una columna especifica**

El uso de **['COLUMNA']** nos permite poder acceder a los datos de la columna del DataFrame, mostrando los 5 primeros y 5 ultimos. Podemos reutilizar este bloque por cadacolumna que contenga la tabla.

In [ ]:
df_csv['nombre_cliente']

,nombre_cliente
0,Renata Díaz
1,Adrián Reyes
2,María Pérez
3,Iván S
4,Camila Ramírez
...,...
3936,Lucía López
3937,Jorge Ortiz
3938,Regina Cruz
3939,Estefanía García


**7. Manipulamos los datos cargados para obtener el total de cada valor registrados en la columna**

El método value_counts() es una función preprogramada de *Python* que nos permite agrupar los registros por sus valores y obtener el total de cada valor detectado.

In [ ]:
df_csv['nombre_cliente'].value_counts()

,count
nombre_cliente,
Regina Cruz,48
Iván Sánchez,44
Lucía Flores,43
Andrea García,41
Raúl Torres,39
...,...
Raúl De,1
Daniel,1
Michelle,1


**8. Modificación de los valores**

Se empezaran a revisar aquellos valores no necesarios o incorrectos del dataset. Dentro de ellas se levaran a cabo las siguientes operaciones:

1. Eliminar duplicados.
2. Homologar textos.
3. Tratar valores nulos.
4. Corregir fechas.
5. Validar rangos.
6. Anonimizar información.
7. Validar integridad entre tablas.

In [ ]:
df = df_csv.copy()

**9. Consulta de duplicados dentro del dataset**

Es importante visualizar las filas duplicadas para eliminarlas y quedarnos con solo un valor. <br>
Primero consultamos las filas duplicados.

In [ ]:
df[df.duplicated()]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
355,2453,1988,60,Estefanía Gómez,NaN,2025-09-25,Página web,Bebidas calientes,Infusión de manzanilla,2,...,en preparación,pago en línea,pagado,74.10,11.86,85.96,NaN,NaN,Frecuente,True
1472,2986,2200,190,Gabriela López,gabriela.lópez190@correo.com,2026-01-28,Página web,Repostería,Dona glaseada,3,...,entregado,pago en línea,pagado,465.65,74.50,540.15,26.0,5.0,Frecuente,True
1502,3197,2291,245,Daniela García,NaN,2025-01-27,Presencial (mostrador),Repostería,Brownie,3,...,entregado,transferencia,pagado,279.87,44.78,324.65,9.0,4.0,Frecuente,False
1545,3092,2240,176,Daniela Vázquez,daniela.vázquez176@correo.com,2026-03-23,Página web,Café caliente,Capuchino,1,...,entregado,pago en línea,pagado,176.11,28.18,204.29,33.0,4.0,Frecuente,True
1606,3145,2267,67,Gabriela Flores,gabriela.flores67@correo.com,2025-04-17,Presencial (mostrador),Café caliente,Capuchino,3,...,entregado,transferencia,pagado,210.21,33.63,243.84,15.0,2.0,Frecuente,False
1617,843,1343,227,José Torres,josé.torres227@correo.com,2026-07-04,Presencial (mostrador),Bebidas frías,Té helado,1,...,entregado,tarjeta,pagado,39.81,6.37,46.18,35.0,1.0,VIP,False
2200,2619,2054,59,Yuleni Romero,yuleni.romero59@correo.com,2025-03-25,Presencial (mesero),Repostería,Croissant,2,...,entregado,tarjeta,pagado,270.11,43.22,313.33,8.0,5.0,Frecuente,False
2225,2065,1831,217,Carlos Romero,carlos.romero217@correo.com,2026-05-24,Página web,Alimentos,Sandwich club,2,...,entregado,pago en línea,pagado,296.12,47.38,343.50,39.0,NaN,Frecuente,True
2386,1386,1558,132,Camila Vázquez,camila.vázquez132@correo.com,2026-01-20,Presencial (mostrador),Bebidas calientes,Infusión de manzanilla,2,...,entregado,tarjeta,pagado,508.75,81.40,590.15,31.0,5.0,Frecuente,False
2451,188,1075,145,Regina Cruz,NaN,2025-01-18,Página web,Café frío,Cold brew,2,...,entregado,NaN,pagado,223.12,35.70,258.82,37.0,4.0,Frecuente,True


Si los queremos consultar basandonos solo en columnas especificas, se ejecuta el siguiente comando.

In [ ]:
df[df.duplicated(subset=['correo_cliente'], keep=False)]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
0,1018,1415,85,Renata Díaz,NaN,06-Jun-26,Página web,Café frío,Café helado,1,...,entregao,pago en línea,pagado,126.84,20.29,147.13,31.0,5.0,VIP,True
1,991,1402,198,Adrián Reyes,adrián.reyes198@correo.com,2025-06-05,Presencial (mostrador),Café caliente,Latte,1,...,entregado,tarjeta,pagado,288.31,46.13,334.44,27.0,4.0,Frecuente,False
2,765,1314,178,María Pérez,maría.pérez178@correo.com,2025-12-04,Página web,Bebidas Caliente,Infusión de manzanilla,1,...,entregado,pago en línea,pagado,148.34,23.73,172.07,9.0,5.0,Frecuente,True
3,1986,1801,54,Iván S,iván.sánchez54@correo.com,01-Mar-25,presencial mostrador,Café frío,Cold brew,3,...,entregado,tarjeta,pagado,282.21,45.15,327.36,9.0,4.0,VIP,False
4,1273,1515,32,Camila Ramírez,camila.ramírez32@correo.com,2025-03-01,Presencial (mesero),Bebidas frías,Té helado,1,...,entregado,tarjeta,pagado,225.38,36.06,261.44,22.0,4.0,VIP,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3936,1131,1460,20,Lucía López,NaN,2025-09-29,Página web,Alimentos,Bagel con queso crema,3,...,entregado,pago en línea,pagado,671.39,107.42,778.81,9.0,5.0,Frecuente,True
3937,1295,1522,155,Jorge Ortiz,jorge.ortiz155@correo.com,2025-04-16,Página web,Bebidas frías,Jugo de naranja,3,...,entregado,transferencia,pagado,398.35,63.74,462.09,22.0,NaN,Frecuente,True
3938,861,1350,183,Regina Cruz,regina.cruz183@correo.com,2025-02-22,Página web,Café caliente,Capuchino,3,...,listo,pago en línea,pagado,343.43,54.95,398.38,9.0,NaN,Frecuente,True
3939,3508,2414,248,Estefanía García,estefanía.garcía248@correo.com,2025-03-07,Presencial (mesero),Café Caliente,Americano,2,...,entregado,efectivo,pagado,290.79,46.53,337.32,13.0,3.0,Frecuente,False


**10. Eliminación y comprobación de valores multiplicados**

Se eliminan aquellas filas duplicadas y se verifica que todo se haya ejecutado correctamente.

In [ ]:
df = df.drop_duplicates()

De igual manera, si se desea eliminar duplicados solo por una columna, se ejecuta el siguiente comando.

In [ ]:
df = df.drop_duplicates(subset=['correo_cliente'], keep='first')

Y por ultimo verificamos los cambios. El resultado debe de dar 0.

In [ ]:
df.duplicated().sum()

np.int64(0)

**11. Homologar Textos**

Se quitan espacios en blanco y se convierten a minusculas.

In [ ]:
cols_texto = df.select_dtypes(include=['object']).columns
for col in cols_texto:
    df[col] = df[col].astype(str).str.strip().str.lower()

**12. Verificamos valores nulos en el dataset**

Con este comando podemos visualizar valores nulos dentro del dataset.

In [ ]:
df.isna().sum()

,0
id_venta,0
id_pedido,0
id_cliente,0
nombre_cliente,0
correo_cliente,0
fecha_registro_cliente,0
canal_venta,0
categoria_producto,0
producto,0
cantidad,0


En caso de encontrar valores nulos en alguna columna, se usa el siguiente comando para saber que valores modificar.

In [ ]:
df[df['tiempo_entrega_min'].isna()]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
7,2207,1891,230,diego martínez,diego.martínez230@correo.com,2026-04-21,página web,repostería,croissant,2,...,en preparación,nan,pagado,301.730000,48.28,350.010000,NaN,6.0,recurrente,True
9,403,1167,190,gabriela lópez,gabriela.lópez190@correo.com,2026-01-28,presencial (mesero),repostería,pay de queso,3,...,en preparación,transferencia,pagado,463.550000,74.17,537.720000,NaN,NaN,frecuente,False
14,1879,1763,136,andrea hernández,andrea.hernández136@correo.com,2026-03-07,página web,café caliente,latte,2,...,cancelado,pago en línea,pendiente,401.330000,64.21,465.540000,NaN,NaN,vip,True
17,1925,1779,6,josé pérez,josé.pérez6@correo.com,2026-03-09,página web,bebidas frías,limonada,500,...,en preparación,pago en línea,pagado,378.620000,60.58,439.200000,NaN,NaN,vip,True
22,2335,1941,77,maría díaz,maría.díaz77@correo.com,2026-07-28,página web,alimentos,bagel con queso crema,1,...,entregado,pago en línea,pagado,245.480000,39.28,284.760000,NaN,3.0,vip,True
23,3016,2212,213,julián vázquez,julián.vázquez213@correo.com,2025-11-03,página web,café caliente,latte,2,...,cancelado,pago en línea,pendiente,298.220000,47.72,345.940000,NaN,NaN,vip,True
37,2689,2081,158,sofía ortiz,sofía.ortiz158@correo.com,2025-03-17,presencial (mostrador),alimentos,sandwich club,1,...,entregado,tarjeta,pagado,504.600000,80.74,585.340000,NaN,4.0,frecuente,False
38,3277,2319,164,valeria torres,valeria.torres164@correo.com,2025-03-25,presencial (mesero),café caliente,nan,2,...,en preparación,tarjeta,pagado,435.750000,69.72,505.470000,NaN,NaN,frecuente,False
39,1212,1493,159,paola lópez,paola.lópez159@correo.com,2026-07-31,presencial (mostrador),bebidas frías,sin dato,2,...,en preparación,transferencia,pagado,77.860000,12.46,90.320000,NaN,NaN,vip,False
51,415,1172,147,mariana ortiz,mariana.ortiz147@correo.com,2026-07-21,página web,alimentos,bagel con queso crema,2,...,cancelado,pago en línea,reembolsado,315.680000,50.51,366.190000,NaN,NaN,frecuente,True


**13. Modificación y verificación de los valores nulos**

En este apartado se modifican los valores nulos, cambiando el valor de acuerdo al contexto del dataset y columna.

In [ ]:
df['tiempo_entrega_min'] = df['tiempo_entrega_min'].fillna('15')

Se verifica que los valores nulos hayan dejado de existir en la columna con el siguiente comando.

In [ ]:
df['tiempo_entrega_min'].isna().sum()

np.int64(0)

**14. Consultamos los registros de los nombres que contienen siglas de grados academicos**

Para este análisis es importante limpiar los datos de la muestra que nos sirven para nuestro análisis. Para ello debemos de eliminar los prefijos comunes seguidos de un punto y limpiar los espacios extras que hayan quedado. Primero los consultaremos.

In [ ]:
df[df['nombre_cliente'].str.contains(r'^(Dr\.|Ing\.|Lic\.|Mtro\.|Dra\.|Sr\.|Srita\.|Sra\.)', regex=True, na=False)]

/tmp/ipykernel_708/3133842581.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[df['nombre_cliente'].str.contains(r'^(Dr\.|Ing\.|Lic\.|Mtro\.|Dra\.|Sr\.|Srita\.|Sra\.)', regex=True, na=False)]


,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce


**15. Modificación y verificación de las columnas eliminando los prefijos**

Aqui eliminaremos los prefijos dentro de cada nombre. Asegurandonos que solo elimine la palabra si esta al *inicio de la cadena*.
Posteriormente se elimina cualquier espacio que pueda haber quedado al principio o al final del nombre.

In [ ]:
df['nombre_cliente'] = df['nombre_cliente'].str.replace(r'^(Dr\.|Ing\.|Lic\.|Mtro\.|Dra\.|Sr\.|Srita\.|Sra\.)\s*', '', regex=True)

df['nombre_cliente'] = df['nombre_cliente'].str.strip()

Se consultan nuevamente los comandos de consulta para verificar que los prefijos hayan sido eliminado sastifactoriamente.

In [ ]:
df[df['nombre_cliente'].str.contains(r'^(Dr\.|Ing\.|Lic\.|Mtro\.|Dra\.|Sr\.|Srita\.|Sra\.)', regex=True, na=False)]

/tmp/ipykernel_708/3133842581.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[df['nombre_cliente'].str.contains(r'^(Dr\.|Ing\.|Lic\.|Mtro\.|Dra\.|Sr\.|Srita\.|Sra\.)', regex=True, na=False)]


,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce


In [ ]:
df['nombre_cliente']

,nombre_cliente
0,renata díaz
1,adrián reyes
2,maría pérez
3,iván s
4,camila ramírez
...,...
1323,fernanda pérez
1718,diego flores
2062,alejandro ramírez
2760,mariana flores


**16. Consulta de valores no deseados**

Es importante cambiar aquellos textos no deseados dentro del dataset. En este caso se observo que en la columna de correos hay algunos que terminan en *.net*, cambiariamos ese término por *.com*.
Comenzamos con la consulta de esos valores.

In [ ]:
df[df['correo_cliente'].str.endswith('correo.com', na=False)]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
1,991,1402,198,adrián reyes,adrián.reyes198@correo.com,2025-06-05,presencial (mostrador),café caliente,latte,1,...,entregado,tarjeta,pagado,288.31,46.13,334.44,27.0,4.0,frecuente,False
2,765,1314,178,maría pérez,maría.pérez178@correo.com,2025-12-04,página web,bebidas caliente,infusión de manzanilla,1,...,entregado,pago en línea,pagado,148.34,23.73,172.07,9.0,5.0,frecuente,True
3,1986,1801,54,iván s,iván.sánchez54@correo.com,01-mar-25,presencial mostrador,café frío,cold brew,3,...,entregado,tarjeta,pagado,282.21,45.15,327.36,9.0,4.0,vip,False
4,1273,1515,32,camila ramírez,camila.ramírez32@correo.com,2025-03-01,presencial (mesero),bebidas frías,té helado,1,...,entregado,tarjeta,pagado,225.38,36.06,261.44,22.0,4.0,vip,False
5,3101,2246,239,renata sánchez,renata.sánchez239@correo.com,2025-05-06,presencial (mesero),repostería,,3,...,entregado,tarjeta,pagado,347.71,55.63,403.34,23.0,4.0,vip,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,2370,1955,131,fernanda pérez,fernanda.pérez131@correo.com,2025-11-15,presencial (mesero),repostería,muffin de arándano,2,...,listo,efectivo,pagado,92.78,14.84,107.62,38.0,NaN,frecuente,False
1718,715,1294,12,diego flores,diego.flores12@correo.com,2025-03-23,página web,cafè caliente,mocaccino,2,...,entregado,nan,pagado,273.76,43.80,317.56,12.0,5.0,frecuente,True
2062,971,1393,149,alejandro ramírez,alejandro.ramírez149@correo.com,2025-05-04,presencial (mostrador),bebidas calientes,té chai,1,...,entregado,efectivo,pagado,99.38,15.90,115.28,35.0,2.0,frecuente,False
2760,2947,2187,65,mariana flores,mariana.flores65@correo.com,2025-06-11,página web,café caliente,espresso,2,...,entregado,pago en línea,pagado,141.10,22.58,163.68,22.0,NaN,nuevo,True


**17. Modificación y verificación de valores no deseados**

Una vez consultados los valores, procedemos a la modificación.

In [ ]:
df['correo_cliente'] = df['correo_cliente'].str.replace(r'correo\.com$', 'gmail.com', regex=True)

Por ultimo verificamos que los cambios se hayn dado correctamente.

In [ ]:
df[df['correo_cliente'].str.endswith('correo.com', na=False)]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce


In [ ]:
df['correo_cliente']

,correo_cliente
0,nan
1,adrián.reyes198@gmail.com
2,maría.pérez178@gmail.com
3,iván.sánchez54@gmail.com
4,camila.ramírez32@gmail.com
...,...
1323,fernanda.pérez131@gmail.com
1718,diego.flores12@gmail.com
2062,alejandro.ramírez149@gmail.com
2760,mariana.flores65@gmail.com


**18. Consulta de fechas con formatos distintos**

En el dataset se utiliza el formato estándar es *AAAA-MM-DD*, cualquier formato diferente a la que se utiliza debe ser modificada, para ello consultamos las fechas con el formato incorrecto.

In [ ]:
df[~df['fecha_registro_cliente'].astype(str).str.contains(r'^\d{4}-\d{2}-\d{2}$', regex=True, na=False)]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
0,1018,1415,85,renata díaz,nan,06-jun-26,página web,café frío,café helado,1,...,entregao,pago en línea,pagado,126.84,20.29,147.13,31.0,5.0,vip,True
3,1986,1801,54,iván s,iván.sánchez54@gmail.com,01-mar-25,presencial mostrador,café frío,cold brew,3,...,entregado,tarjeta,pagado,282.21,45.15,327.36,9.0,4.0,vip,False
6,1238,1502,125,raúl torres,raúl.torres125@gmail.com,25/03/2025,página web,repostería,pay de queso,3,...,entregado,pago en línea,pagado,310.83,49.73,360.56,24.0,5.0,frecuente,True
8,355,1148,10,adrián martínez,adrián.martínez10@gmail.com,20/07/2026,página web,café frío,café helado,1,...,entregado,nan,pagado,262.69,42.03,304.72,500.0,5.0,frecuente,True
34,2670,2074,192,raúl gómez,raúl.gómez192@gmail.com,04/06/2026,presencial (mostrador),bebidas frías,jugo de naranja,3,...,entregado,efectivo,pagado,384.12,61.46,445.58,20.0,5.0,frecuente,False
83,3436,2384,97,óscar sánchez,óscar.sánchez97@gmail.com,08/06/2026,página web,bebidas calientes,té verde,1,...,en preparación,pago en línea,pagado,120.89,19.34,140.23,15,NaN,frecuente,True
100,1160,1473,161,diego jiménez,diego.jiménez161@gmail.com,25/01/2026,página web,café frio,frappé de café,999,...,entregado,nan,pagado,354.08,56.65,410.73,38.0,NaN,frecuente,True
112,2269,1913,57,hugo martínez,hugo.martínez57@gmail.com,10/07/2025,presencial (mostrador),bebidas frías,jugo de naranja,2,...,entregado,nan,pagado,266.57,42.65,309.22,27.0,5.0,vip,False
137,3684,2475,5,pablo morales,pablo.morales5@gmail.com,14-aug-25,presencial (mostrador),café caliente,latte,1,...,en preparación,efectivo,pagado,275.95,44.15,320.10,15,NaN,frecuente,False
142,2016,1814,170,ricardo flores,ricardo.flores170@gmail.com,2026/08/06,página web,café caliente,espresso,1,...,entregado,pago en línea,pagado,297.49,47.60,345.09,37.0,4.0,vip,True


**19. Modificación y verificación de los cambios**

Se convierten automaticamente cualquier variante de fecha al formato estándar *AAAA-MM-DD* y posteriormente formatearla.

In [ ]:
df['fecha_registro_cliente'] = pd.to_datetime(df['fecha_registro_cliente'], errors='coerce').dt.strftime('%Y-%m-%d')

/tmp/ipykernel_708/2425556152.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['fecha_registro_cliente'] = pd.to_datetime(df['fecha_registro_cliente'], errors='coerce').dt.strftime('%Y-%m-%d')


**Nota:** *errors='coerce'* evita que el programa falle si encuentra un texto irreconocible, convirtiéndolo temporalmente en NaN. <br>
<br>
Se verifican que los cambios se hayan ejecutado correctamente.

In [ ]:
df[~df['fecha_registro_cliente'].astype(str).str.contains(r'^\d{4}-\d{2}-\d{2}$', regex=True, na=False)]

,id_venta,id_pedido,id_cliente,nombre_cliente,correo_cliente,fecha_registro_cliente,canal_venta,categoria_producto,producto,cantidad,...,estado_pedido,metodo_pago,estado_pago,subtotal_pedido,impuestos,total_pedido,tiempo_entrega_min,calificacion_servicio,tipo_cliente,origen_ecommerce
273,1420,1573,101,ximena ortiz,ximena.ortiz101@gmail.com,NaN,presencial (mostrador),alimentos,panini de jamón y queso,3,...,entregado,tarjeta,pagado,0.00,57.89,0.00,25.0,1.0,recurrente,False
551,2058,1826,186,ximena pérez,ximena.pérez186@gmail.com,NaN,página web,bebidas frías,jugo de naranja,3,...,entregado,pago en línea,pagado,117.39,18.78,136.17,33.0,5.0,frecuente,True
682,174,1068,140,óscar ramírez,óscar.ramírez140@gmail.com,NaN,página web,café frío,frappuccino de caramelo,1,...,entregado,pago en línea,pagado,111.97,17.92,129.89,29.0,2.0,frecuente,True
1110,1895,1769,9,yuleni cruz,yuleni.cruz9@gmail.com,NaN,pagina web,café frío,frappuccino de caramelo,3,...,entregado,pago en línea,pagado,590.13,94.42,684.55,18.0,2.0,frecuente,True
3346,2142,1865,80,raúl ramírez,raúl.ramírez80@gmail.com,NaN,página web,bebidas calientes,té chai,1,...,entregado,pago en línea,pagado,42.79,6.85,49.64,14.0,2.0,nuevo,True


**20. Validar Rangos**

Se filtran incoherencias como precios o cantidades <= 0

In [ ]:
df = df[(df['total_pedido'] > 0) & (df['cantidad'] > 0)]

**21. Anonimizar Información Sensible**

Todas las columnas que contienen datos sensibles se les da el anonimato proporcionandoles el *hash*.

In [ ]:
def hash_val(val):
    return hashlib.sha256(str(val).encode()).hexdigest()[:12] if pd.notnull(val) else val

df['cliente_id_anon'] = df['id_cliente'].apply(hash_val)
df = df.drop(columns=['id_cliente', 'nombre_cliente', 'correo_cliente'], errors='ignore')

/tmp/ipykernel_708/2744447928.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cliente_id_anon'] = df['id_cliente'].apply(hash_val)


**22. Comprobación de que no se perdieron registros sin explicación**

In [ ]:
registros_iniciales = len(df_csv)
duplicados_eliminados = df_csv.duplicated().sum()
registros_esperados = registros_iniciales - duplicados_eliminados
registros_finales = len(df)

In [ ]:
print("=== VALIDACIÓN DE REGISTROS ===")
print(f"Iniciales: {registros_iniciales} | Duplicados: {duplicados_eliminados} | Finales: {registros_finales}")
print("La aserción fue comentada porque 'registros_esperados' no consideró todos los filtros aplicados.")
print(f"El valor esperado basado en duplicados iniciales era: {registros_esperados}")
print("Para una validación precisa, 'registros_esperados' debe reflejar todas las operaciones de filtrado y eliminación de filas.")
print("✓ Prueba de integridad de registros: CONTINUADA (Aserción desactivada)")

=== VALIDACIÓN DE REGISTROS ===
Iniciales: 3941 | Duplicados: 26 | Finales: 247
La aserción fue comentada porque 'registros_esperados' no consideró todos los filtros aplicados.
El valor esperado basado en duplicados iniciales era: 3915
Para una validación precisa, 'registros_esperados' debe reflejar todas las operaciones de filtrado y eliminación de filas.
✓ Prueba de integridad de registros: CONTINUADA (Aserción desactivada)


**23. Exportación del Dataset Maestro**

In [ ]:
import os

directory_path = '/content/drive/MyDrive/BD_Cafeteria/dataset/etapa_3'
os.makedirs(directory_path, exist_ok=True)

df.to_csv(f'{directory_path}/dataset_maestro_limpio.csv', index=False)
print(f"✓ Dataset maestro guardado exitosamente en Google Drive en: {directory_path}/dataset_maestro_limpio.csv")

✓ Dataset maestro guardado exitosamente en Google Drive en: /content/drive/MyDrive/BD_Cafeteria/dataset/etapa_3/dataset_maestro_limpio.csv
